# Implementasi Bab 9 — Ensemble Learning

Decision Tree tunggal dibandingkan dengan Random Forest, AdaBoost, dan Gradient Boosting pada dataset Wine [ML-MODUL-01, hlm. 63–64].


In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier

SEED = 42
random.seed(SEED)
np.random.seed(SEED)


In [ ]:
data = load_wine(as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=SEED, stratify=data.target
)
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=SEED),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, oob_score=True, random_state=SEED, n_jobs=-1
    ),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=SEED),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100, learning_rate=0.1, random_state=SEED
    ),
}
for candidate in models.values():
    candidate.fit(X_train, y_train)


In [ ]:
random_forest = models["Random Forest"]
print("OOB Random Forest:", round(random_forest.oob_score_, 3))
holdout_rows = []
for name, candidate in models.items():
    score = accuracy_score(y_test, candidate.predict(X_test))
    holdout_rows.append((name, score))
display(pd.DataFrame(holdout_rows, columns=["model", "test_accuracy"]).sort_values("test_accuracy", ascending=False))


In [ ]:
importances = random_forest.feature_importances_
indices = np.argsort(importances)[::-1][:8]
plt.barh(
    [data.feature_names[index] for index in indices][::-1],
    importances[indices][::-1],
)
plt.xlabel("Feature Importance")
plt.title("Feature Importance — Random Forest")
plt.show()


## Latihan 1: pengaruh n_estimators

OOB score menggunakan sampel yang tidak terambil pada bootstrap masing-masing pohon sebagai estimasi internal [ML-MODUL-01, hlm. 64].


In [ ]:
estimator_rows = []
for n_estimators in [10, 50, 100, 300, 500]:
    candidate = RandomForestClassifier(
        n_estimators=n_estimators, oob_score=True, random_state=SEED, n_jobs=-1
    ).fit(X_train, y_train)
    estimator_rows.append((n_estimators, candidate.oob_score_))
estimator_frame = pd.DataFrame(estimator_rows, columns=["n_estimators", "oob"])
display(estimator_frame)
selected_n_estimators = int(
    estimator_frame.sort_values("oob", ascending=False).iloc[0]["n_estimators"]
)
selected_forest = RandomForestClassifier(
    n_estimators=selected_n_estimators, random_state=SEED, n_jobs=-1
).fit(X_train, y_train)
final_test_accuracy = accuracy_score(y_test, selected_forest.predict(X_test))
print("n_estimators terpilih:", selected_n_estimators)
print("Akurasi test final:", round(final_test_accuracy, 3))
estimator_frame.plot(x="n_estimators", y="oob", marker="o")
plt.ylim(0.7, 1.02)
plt.show()


## Latihan 2: Stratified k-Fold CV

Empat model dibandingkan dengan split dan metrik `f1_macro` yang sama agar adil [ML-MODUL-01, hlm. 64].


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_rows = []
for name, candidate in models.items():
    scores = cross_val_score(candidate, data.data, data.target, cv=cv, scoring="f1_macro", n_jobs=-1)
    cv_rows.append((name, scores.mean(), scores.std()))
display(pd.DataFrame(cv_rows, columns=["model", "mean_f1_macro", "std"]).sort_values("mean_f1_macro", ascending=False))


## Kesimpulan

Ensemble mengurangi kelemahan satu pohon melalui agregasi atau pembelajaran bertahap. Performa, stabilitas CV, biaya komputasi, dan interpretabilitas perlu dipertimbangkan bersama.
